# Create Virginia Tech ETD Dataset

This notebook documents and runs the scripted workflow for building the Virginia Tech ETD dataset from VTechWorks OAI-PMH DIM records. It harvests the doctoral and master's thesis sets, extracts the requested metadata fields including `date_issued`, filters out records with empty abstracts by default, normalizes department labels for reporting, creates an 80/20 department-stratified train/test split, writes a department summary CSV for Chapter 3 reporting, and exports audit tables showing both approved department merges and any residual fuzzy-match candidates.


## Endpoints and Parameters

This notebook uses the Virginia Tech VTechWorks OAI-PMH DIM endpoints below:

- Doctoral dissertations: `https://vtechworks.lib.vt.edu/server/oai/request?verb=ListRecords&metadataPrefix=dim&set=col_10919_11041`
- Master's theses: `https://vtechworks.lib.vt.edu/server/oai/request?verb=ListRecords&metadataPrefix=dim&set=col_10919_9291`

The run documented here uses these operational settings:

- `test_fraction = 0.2`
- `seed = 200`
- `skip_harvest = True` when rebuilding from previously downloaded XML in `data/raw/vtechworks_oai`
- `keep_empty_abstracts = False` so rows with no abstract are excluded before export
- output files:
  - `vt_etds_all.csv`
  - `vt_etds_train.csv`
  - `vt_etds_test.csv`
  - `vt_etds_department_summary.csv`
  - `vt_etds_department_mapping_audit.csv`
  - `vt_etds_department_similarity_candidates.csv`

Current ETD row schema highlights:

- `date_issued` comes from `dc.date.issued`
- `oai_datestamp` is not retained
- `set_spec` is not retained
- `department_normalized` is the stratification column used for the split


In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SCRIPT = PROJECT_ROOT / 'scripts' / 'create_etd_dataset.py'

DOCTORAL_URL = (
    'https://vtechworks.lib.vt.edu/server/oai/request'
    '?verb=ListRecords&metadataPrefix=dim&set=col_10919_11041'
)
MASTERS_URL = (
    'https://vtechworks.lib.vt.edu/server/oai/request'
    '?verb=ListRecords&metadataPrefix=dim&set=col_10919_9291'
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'vtechworks_oai'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'vtechworks'

COMBINED_NAME = 'vt_etds_all.csv'
TRAIN_NAME = 'vt_etds_train.csv'
TEST_NAME = 'vt_etds_test.csv'
DEPARTMENT_SUMMARY_NAME = 'vt_etds_department_summary.csv'
DEPARTMENT_MAPPING_NAME = 'vt_etds_department_mapping_audit.csv'
DEPARTMENT_CANDIDATES_NAME = 'vt_etds_department_similarity_candidates.csv'

TEST_FRACTION = 0.2
SEED = 200
SKIP_HARVEST = True
KEEP_EMPTY_ABSTRACTS = False

config = {
    'project_root': PROJECT_ROOT,
    'script': SCRIPT,
    'doctoral_url': DOCTORAL_URL,
    'masters_url': MASTERS_URL,
    'raw_dir': RAW_DIR,
    'processed_dir': PROCESSED_DIR,
    'combined_name': COMBINED_NAME,
    'train_name': TRAIN_NAME,
    'test_name': TEST_NAME,
    'department_summary_name': DEPARTMENT_SUMMARY_NAME,
    'department_mapping_name': DEPARTMENT_MAPPING_NAME,
    'department_candidates_name': DEPARTMENT_CANDIDATES_NAME,
    'test_fraction': TEST_FRACTION,
    'seed': SEED,
    'skip_harvest': SKIP_HARVEST,
    'keep_empty_abstracts': KEEP_EMPTY_ABSTRACTS,
}

for key, value in config.items():
    print(f'{key}: {value}')


## Exact Script Invocation

The next cell constructs the exact command used to build the ETD dataset. Because the URLs are included explicitly, the command is self-contained rather than relying on script defaults. With `SKIP_HARVEST = True`, the command reuses the downloaded XML pages already stored in `data/raw/vtechworks_oai`; set it to `False` to re-harvest from VTechWorks.


In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    '--doctoral-url', DOCTORAL_URL,
    '--masters-url', MASTERS_URL,
    '--raw-dir', str(RAW_DIR),
    '--processed-dir', str(PROCESSED_DIR),
    '--combined-name', COMBINED_NAME,
    '--train-name', TRAIN_NAME,
    '--test-name', TEST_NAME,
    '--department-summary-name', DEPARTMENT_SUMMARY_NAME,
    '--department-mapping-name', DEPARTMENT_MAPPING_NAME,
    '--department-candidates-name', DEPARTMENT_CANDIDATES_NAME,
    '--test-fraction', str(TEST_FRACTION),
    '--seed', str(SEED),
]

if SKIP_HARVEST:
    cmd.append('--skip-harvest')
if KEEP_EMPTY_ABSTRACTS:
    cmd.append('--keep-empty-abstracts')

print(shlex.join(cmd))


## How the Train/Test Split Is Obtained

The split is created by `scripts/create_etd_dataset.py`, not by notebook-local logic. The relevant workflow is:

1. Harvest all paged OAI-PMH DIM responses from the doctoral and master's endpoints.
2. Parse the requested metadata fields from each `oai:record`, including `date_issued` from `dc.date.issued`.
3. Deduplicate records by `(oai_identifier, uri)`.
4. Filter out rows with empty abstracts unless `--keep-empty-abstracts` is set.
5. Build `department_normalized` by collapsing obvious department-label variants such as `&` vs `and`, malformed `and#38;`, parenthetical qualifiers, approved candidate merges, and a small set of conservative manual overrides.
6. Run a deterministic 80/20 split stratified on `department_normalized` using `seed = 200`.
7. Keep singleton departments in the training split so the test split does not contain unseen single-row strata.

This means the train/test split is stratified across normalized departments rather than the raw department strings exposed in VTechWorks metadata.


In [ ]:
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)


## Post-run Validation

The next cell checks the row counts of the generated CSVs, confirms that abstract-less rows were removed, verifies the presence of `date_issued`, summarizes the resulting department-stratified split, and reports the size of the department audit artifacts.


In [ ]:
import csv
from collections import Counter

all_path = PROCESSED_DIR / COMBINED_NAME
train_path = PROCESSED_DIR / TRAIN_NAME
test_path = PROCESSED_DIR / TEST_NAME
summary_path = PROCESSED_DIR / DEPARTMENT_SUMMARY_NAME
mapping_path = PROCESSED_DIR / DEPARTMENT_MAPPING_NAME
candidate_path = PROCESSED_DIR / DEPARTMENT_CANDIDATES_NAME

for path in [all_path, train_path, test_path, summary_path, mapping_path, candidate_path]:
    print(path)
    print(f'  exists: {path.exists()}')

with all_path.open(newline='', encoding='utf-8') as f:
    all_rows = list(csv.DictReader(f))
with train_path.open(newline='', encoding='utf-8') as f:
    train_rows = list(csv.DictReader(f))
with test_path.open(newline='', encoding='utf-8') as f:
    test_rows = list(csv.DictReader(f))
with summary_path.open(newline='', encoding='utf-8') as f:
    department_rows = list(csv.DictReader(f))
with mapping_path.open(newline='', encoding='utf-8') as f:
    mapping_rows = list(csv.DictReader(f))
with candidate_path.open(newline='', encoding='utf-8') as f:
    candidate_rows = list(csv.DictReader(f))

def blank_count(rows, column):
    return sum(1 for row in rows if not (row.get(column) or '').strip())

print('\nRow counts')
print(f'  combined: {len(all_rows)}')
print(f'  train:    {len(train_rows)}')
print(f'  test:     {len(test_rows)}')
print(f'  departments: {len(department_rows)}')
print(f'  mapping audit rows: {len(mapping_rows)}')
print(f'  fuzzy candidate rows: {len(candidate_rows)}')

print('\nBlank-field checks')
print(f"  combined blank abstracts: {blank_count(all_rows, 'abstract')}")
print(f"  train blank abstracts:    {blank_count(train_rows, 'abstract')}")
print(f"  test blank abstracts:     {blank_count(test_rows, 'abstract')}")
print(f"  combined blank date_issued: {blank_count(all_rows, 'date_issued')}")
print(f"  combined blank normalized departments: {blank_count(all_rows, 'department_normalized')}")

source_counts = Counter(row['source_set'] for row in all_rows)
print('\nSource-set counts (combined)')
for source_set, count in sorted(source_counts.items()):
    print(f'  {source_set}: {count}')

train_departments = Counter(row['department_normalized'] for row in train_rows)
test_departments = Counter(row['department_normalized'] for row in test_rows)
print('\nDepartment coverage')
print(f'  unique normalized departments in train: {len(train_departments)}')
print(f'  unique normalized departments in test:  {len(test_departments)}')
print(f'  singleton departments in train: {sum(1 for v in train_departments.values() if v == 1)}')

print('\nSchema spot check')
print(sorted(all_rows[0].keys()))


## Abstract Length Analysis

The next cell computes ETD abstract lengths in both words and BERT tokens so the ETD corpus can be compared directly with the Scopus abstract corpus. It reports the mean, median, and standard deviation for both views and plots the token-length distribution.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

df_etd = pd.DataFrame(all_rows).copy()
df_etd['Abstract Word Length'] = df_etd['abstract'].fillna('').apply(lambda x: len(str(x).split()))
df_etd['Abstract Token Length'] = df_etd['abstract'].fillna('').apply(lambda x: len(tokenizer.tokenize(str(x))))

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_etd, x=df_etd.index, y='Abstract Token Length', marker='o', s=18, linewidth=0)
plt.title('Distribution of Token Lengths in VT ETD Abstracts')
plt.xlabel('ETD Index')
plt.ylabel('Token Length')
plt.show()

word_median = df_etd['Abstract Word Length'].median()
word_mean = df_etd['Abstract Word Length'].mean()
word_std = df_etd['Abstract Word Length'].std()

token_median = df_etd['Abstract Token Length'].median()
token_mean = df_etd['Abstract Token Length'].mean()
token_std = df_etd['Abstract Token Length'].std()

print(f'Median Abstract Word Length: {word_median}')
print(f'Mean Abstract Word Length: {word_mean}')
print(f'Standard Deviation of Abstract Word Length: {word_std}')
print(f'Median Abstract Token Length: {token_median}')
print(f'Mean Abstract Token Length: {token_mean}')
print(f'Standard Deviation of Abstract Token Length: {token_std}')


## Future Work

A possible next step for ETD department normalization is to use embedding-based retrieval against the VT major list to surface additional candidate mappings. The current pipeline keeps canonicalization deterministic and auditable; embeddings would be used, if at all, as a review aid rather than an automatic replacement for explicit mappings.
